<a href="https://colab.research.google.com/github/Nick97382000/Lettura_bilanci/blob/main/Lettura%20bilanci.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Lettura bilanci

In [2]:
!git clone https://github.com/Nick97382000/Lettura_bilanci
!pip install pdfplumber

Cloning into 'Lettura_bilanci'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 56 (delta 19), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 29.84 MiB | 9.98 MiB/s, done.
Resolving deltas: 100% (19/19), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 94.9 MB/s eta 0:00:00


In [3]:
import PyPDF2
import pdfplumber
import pandas as pd

pdf_path_2= "Lettura_bilanci/deposito bilanci/Kirey - 2024.pdf"

with open(pdf_path_2, "rb") as file:
    reader = PyPDF2.PdfReader(file)  # Read the PDF
    text = ""
    for page in reader.pages:
        text += page.extract_text() + ""

print(text[:100])

with pdfplumber.open(pdf_path_2) as pdf:
    page = pdf.pages[2]
    tables = page.extract_tables()

    if tables:
        t = tables[0]
        print(t.shape)
        if t:
            print("KO")
        t_clean = [row for row in t if any(cell is not None for cell in row)]
        max_cols = max(len(row) for row in t_clean)
        t_uniform = [row + [None] * (max_cols - len(row)) for row in t_clean]

        df= pd.DataFrame(t_uniform)
        print(df.shape)
        print(df)




ModuleNotFoundError: No module named 'PyPDF2'

In [4]:
import pdfplumber
import pandas as pd

def trova_tabella_bilancio(pdf_path, keywords = ["immobilizzazioni immateriali",
                                                 "crediti",
                                                 "immobilizzazioni materiali",
                                                 "immobilizzazioni finanziarie"]):
    with pdfplumber.open(pdf_path) as pdf:
        pagina_start = None
        for i, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            for ii, t in enumerate(tables):
                testo = " ".join(
                    str(cell).lower()
                    for row in t
                    for cell in row
                    if cell
                )
                if all(kw in testo for kw in keywords):
                    pagina_start = i
                    break;
            if pagina_start is not None:
                break

        if pagina_start is None:
            return None

        tutte_le_righe = []
        intestazione = None

        for i in range (pagina_start, len(pdf.pages)):
                tables = pdf.pages[i].extract_tables()

                if not tables:
                    break

                for t in tables:
                    righe = [r for r in t if any(c for c in r)]
                    tutte_le_righe.extend(righe)

        if tutte_le_righe:
            n_colonne = max(len(r) for r in tutte_le_righe)
            tutte_le_righe = [r + [None]*(n_colonne - len(r)) for r in tutte_le_righe]

            df = pd.DataFrame(tutte_le_righe)
            return df
    return None


pdf_path_2= "Lettura_bilanci/deposito bilanci/Conserve Italia - Civilistico e Consolidato - 2024.pdf"
tabella = trova_tabella_bilancio(pdf_path_2)

print(tabella)

                                                     0  \
0                          STATO PATRIMONIALE - ATTIVO   
1                                                   A)   
2                                                        
3                                                        
4                                                   B)   
..                                                 ...   
299                                                      
300                                                      
301                                                      
302                                                      
303  Stato patrimoniale\nConto economico\nRendicont...   

                                                     1  \
0                                                 None   
1     CREDITI VERSO SOCI PER VERSAMENTI ANCORA DOVUTI:   
2    Quote capitale sociale sottoscritte e non versate   
3    TOTALE CREDITI V/SOCI PER VERSAMENTI ANCORA DO...   
4            

In [9]:
import pandas as pd

def summary_tabella(tabella):
    labels_dict = {
        "immobilizzazioni_immateriali": [
            "Totale immobilizzazioni immateriali",
            "TOTALE IMMOBILIZZAZIONI IMMATERIALI"
        ],
        "immobilizzazioni_materiali": [
            "Totale immobilizzazioni materiali",
            "TOTALE IMMOBILIZZAZIONI MATERIALI"
        ],
        "immobilizzazioni_finanziarie": [
            "Totale immobilizzazioni finanziarie",
            "TOTALE IMMOBILIZZAZIONI FINANZIARIE"
        ],
        "patrimonio_netto": [
            "Totale patrimonio netto",
            "TOTALE PATRIMONIO NETTO"
        ],
        "liq_inizio": [
            "Totale disponibilità liquide a inizio esercizio",
            "Totale disponibilità liquide all'inizio dell'esercizio",
            "Totale disponibilità liquide a inizio periodo"
        ],
        "debiti_banche": [
            "Totale debiti verso banche",
            "Debiti verso banche",
            "TOTALE DEBITI VERSO BANCHE"
        ],
        "differenza_valore_costi": [
            "Differenza tra valore e costi della produzione",
            "Totale differenza tra valore e costi della produzione"
        ],
        "oneri_finanziari": [
            "Totale interessi e altri oneri finanziari",
            "Totale interessi ed altri oneri finanziari",
            "Interessi e altri oneri finanziari"
        ],
        "ammortamenti_svalutazioni": [
            "Totale ammortamenti e svalutazioni",
            "Ammortamenti e svalutazioni"
        ],
        "flusso_operativo": [
            "Flusso finanziario dell'attività operativa",
            "Totale flusso finanziario dell'attività operativa",
            "Flusso finanziario derivante dall'attività operativa"
        ],
        "flusso_investimento": [
            "Totale flusso finanziario dell'attività di investimento",
            "Flusso finanziario dell'attività di investimento",
            "Flusso finanziario derivante dall'attività di investimento"
        ],
        "flusso_finanziamento": [
            "Flusso finanziario dell'attività di finanziamento (C)",
            "Totale flusso finanziario dell'attività di finanziamento",
            "Flusso finanziario dell'attività di finanziamento"
        ]
    }

    if tabella.shape[1] == 3:
        col_label = 0
        cols_valori = [1, 2]
        date = tabella.iloc[0, 1:3].tolist()

    elif tabella.shape[1] == 5:
        col_label = 1
        cols_valori = [2, 3]
        date = tabella.iloc[0, 2:4].tolist()

    else:
        return pd.DataFrame()

    def riga_valida(valori):
        for v in valori:
            if pd.notna(v) and str(v).strip() != "":
                return True
        return False

    righe = {}

    for nome_output, aliases in labels_dict.items():
        trovato = False

        for testo_da_cercare in aliases:
            mask = tabella.iloc[:, col_label].astype(str).str.contains(
                testo_da_cercare,
                case=False,
                na=False,
                regex=False
            )

            if mask.any():
                possibili_righe = tabella.loc[mask, cols_valori]

                for _, row in possibili_righe.iterrows():
                    valori = row.tolist()

                    if riga_valida(valori):
                        righe[nome_output] = valori
                        trovato = True
                        break

            if trovato:
                break

        if not trovato:
            righe[nome_output] = [None] * len(cols_valori)

    return pd.DataFrame.from_dict(righe, orient="index", columns=date)


In [10]:
import pandas as pd
import numpy as np

def convert_bilancio_italiano(df):
    """Converte bilancio italiano → numeri"""

    # 1. Sostituisci parentesi TONDO (solo per valori negativi)
    df = df.replace({r'\(': '-', r'\)': ''}, regex=True)

    # 2. FUNZIONE per formato italiano: punti→niente, virgola→punto
    def to_numeric_it(x):
        if pd.isna(x) or x == '':
            return np.nan
        x = str(x).strip()
        # Rimuovi punti (migliaia), sostituisci virgola con punto
        x = x.replace('.', '').replace(',', '.')
        return pd.to_numeric(x, errors='coerce')

    # 3. Applica a tutto il DF
    return df.applymap(to_numeric_it)

# USO


In [11]:
import glob
import os
pdf_files = glob.glob("Lettura_bilanci/deposito bilanci/*.pdf")

risultati = {}
summary = {}

for pdf_path in pdf_files:
    nome = os.path.basename(pdf_path).replace(".pdf", "")
    tabella = trova_tabella_bilancio(pdf_path)
    if tabella is not None:
      summary[nome]=convert_bilancio_italiano(summary_tabella(tabella))
print(summary)

/tmp/ipykernel_2047/3462529323.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(to_numeric_it)
/tmp/ipykernel_2047/3462529323.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(to_numeric_it)
/tmp/ipykernel_2047/3462529323.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(to_numeric_it)
/tmp/ipykernel_2047/3462529323.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(to_numeric_it)
/tmp/ipykernel_2047/3462529323.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(to_numeric_it)


{'Conserve Italia - Civilistico e Consolidato - 2024':                               30 giugno 2025  30 giugno 2024
immobilizzazioni_immateriali        44577002        47325342
immobilizzazioni_materiali         342717358       306685238
immobilizzazioni_finanziarie       103068871       106983634
patrimonio_netto                   305929558       300611685
liq_inizio                         106847278       115897075
debiti_banche                      191590518       168282068
differenza_valore_costi             22431477        29731605
oneri_finanziari                   -18826861       -21543220
ammortamenti_svalutazioni          -31210733       -30398405
flusso_operativo                    50640501        47095488
flusso_investimento                -60736793       -35519602
flusso_finanziamento                30858833       -20625683, 'Volpato Industrie - Consolidato - 2024':                               31-12-2024  31-12-2023
immobilizzazioni_immateriali    21393216    21290799
imm

/tmp/ipykernel_2047/3462529323.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(to_numeric_it)


In [8]:
print(convert_bilancio_italiano(summary["IFB SRL - 2024"]))

                                                31-12-2024  31-12-2023
Totale patrimonio netto                           10269303     7759354
Differenza tra valore e costi della produzione      -28556      -61995
Totale interessi e altri oneri finanziar             31290       10828
Totale ammortamenti e svalutazioni                    3999         178


/tmp/ipykernel_2047/3462529323.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(to_numeric_it)
